Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
BASE_DIR    = "/content/drive/MyDrive/Demo/CS4182_NLP/RAG_University"
DOCS_DIR    = os.path.join(BASE_DIR, "documents")
CHROMA_DIR  = os.path.join(BASE_DIR, "chroma_db")

os.makedirs(DOCS_DIR,   exist_ok=True)
os.makedirs(CHROMA_DIR, exist_ok=True)
print("Directories ready.")

Directories ready.


Installing dependencies

In [ ]:
!pip install -q langchain langchain-community langchain-huggingface chromadb sentence-transformers transformers accelerate huggingface_hub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 74.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 77.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.

In [ ]:
import langchain
import chromadb
import transformers
import sentence_transformers
import huggingface_hub

print("All imports successful!")

All imports successful!


Load and chunk documents

In [ ]:
!pip install -q pypdf docx2txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 346.6/346.6 kB 12.0 MB/s eta 0:00:00


In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
import os
from langchain_community.document_loaders import (
    DirectoryLoader,
    TextLoader,
    PyPDFLoader,
    Docx2txtLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

DOCS_DIR = "/content/drive/MyDrive/Demo/CS4182_NLP/RAG_University/documents"

# Load pdf, txt and docx files if there any
loaders = [
    DirectoryLoader(DOCS_DIR, glob="**/*.txt",  loader_cls=TextLoader,
                    loader_kwargs={"encoding": "utf-8"}, show_progress=True),
    DirectoryLoader(DOCS_DIR, glob="**/*.pdf",  loader_cls=PyPDFLoader,
                    show_progress=True),
    DirectoryLoader(DOCS_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader,
                    show_progress=True),
]

raw_docs = []
for loader in loaders:
    try:
        docs = loader.load()
        raw_docs.extend(docs)
        print(f"Loaded {len(docs)} docs from {loader.__class__.__name__}")
    except Exception as e:
        print(f"Loader error: {e}")

print(f"\nTotal documents loaded: {len(raw_docs)}")

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ".", " "],
)
chunks = splitter.split_documents(raw_docs)
print(f"Total chunks: {len(chunks)}")

/tmp/ipykernel_1214/617061451.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import (
100%|██████████| 1/1 [00:00<00:00,  3.66it/s]


Loaded 1 docs from DirectoryLoader


100%|██████████| 7/7 [00:02<00:00,  3.02it/s]


Loaded 14 docs from DirectoryLoader


0it [00:00, ?it/s]

Loaded 0 docs from DirectoryLoader

Total documents loaded: 15
Total chunks: 33


Build the chroma vector store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading embedding model …")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("Building Chroma vector store …")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="university_docs",
)

print(f"Vector store built with {vectorstore._collection.count()} vectors.")
print(f"Persisted to: {CHROMA_DIR}")

Loading embedding model …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building Chroma vector store …
Vector store built with 66 vectors.
Persisted to: /content/drive/MyDrive/Demo/CS4182_NLP/RAG_University/chroma_db


Build the RAG chain

In [ ]:
!pip list | grep langchain

langchain                                1.3.6
langchain-classic                        1.0.8
langchain-community                      0.4.2
langchain-core                           1.4.7
langchain-huggingface                    1.2.2
langchain-protocol                       0.0.16
langchain-text-splitters                 1.1.2


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_huggingface import HuggingFacePipeline
import torch


MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"  # ~1 GB; good for Colab free
# MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct" # ~3 GB — better quality

print(f"Loading tokenizer and model: {MODEL_ID} …")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
    trust_remote_code=True,
)

# Wrap in a text-generation pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.1,
    return_full_text=False,   # return only the generated part, not the prompt
)

# Wrap the pipeline as a LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)
print("LLM ready.")


Loading tokenizer and model: Qwen/Qwen2.5-0.5B-Instruct …


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LLM ready.


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Retriever: fetch top-3 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},
)

# Prompt template
RAG_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful university assistant for the University of Peradeniya.
Use ONLY the information in the context below to answer the question.
If the answer is not in the context, say "I don't have that information in my knowledge base."
Keep your answer clear and concise.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build the RAG chain using modern LCEL syntax
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")

RAG chain ready.


Query the RAG system

In [ ]:
def ask(question: str):
    """Run a question through the RAG chain and print the result neatly."""
    print("=" * 65)
    print(f"Q: {question}")
    print("-" * 65)

    # Get answer
    answer = rag_chain.invoke(question)
    answer = answer.split("Answer:")[-1].strip()
    print(f"A: {answer}")

    # Get sources separately
    print("\n── Retrieved sources ──")
    docs = retriever.invoke(question)
    for i, doc in enumerate(docs, 1):
        src = doc.metadata.get("source", "unknown").split("/")[-1]
        snippet = doc.page_content[:120].replace("\n", " ")
        print(f"  [{i}] {src}: {snippet}…")
    print()

# ── Test questions ──────────────────────────────────────────────────────────
ask("What are the admission requirements for postgraduate study?")
ask("How do I register for courses at the University of Peradeniya?")
ask("What is the grading scale used in the Faculty of Engineering?")
ask("What library services are available to postgraduate students?")
ask("How can I access IEEE Xplore from off campus?")
ask("What financial aid options are available for students?")
ask("What software licenses are available to students for free?")
ask("What is the process for thesis submission?")

Q: What are the admission requirements for postgraduate study?
-----------------------------------------------------------------


[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the docum

A: The admission requirements for postgraduate study include holding a recognized Bachelor's degree with at least a Second Class Lower (2.2) or GPA equivalent, as well as submitting a research proposal and two academic references. Additionally, applicants must meet an English proficiency requirement of either IELTS 6.5 or TOEFL iBT 90. These requirements ensure that students have the necessary qualifications and skills to pursue their studies. It's important to note that these requirements may vary slightly depending on the specific program or field of study being applied for. Please refer to the official website for the most up-to-date information. If you need further clarification on any aspect of this process, feel free to ask! [I don't have that information in my knowledge base.]

── Retrieved sources ──
  [1] admission_requirments.txt: Postgraduate Admission: - Applicants must hold a recognised Bachelor's degree with at least a Second Class   Lower (2.2)…
  [2] admission_requirmen

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: To register for courses at the University of Peradeniya, you need to log into the Student Information System (SIS) at sis.pdn.ac.lk with your university email and password. Then, navigate to the "Course Registration" section within the Academic tab. This process allows you to apply for admission to various courses offered by the university during the semester system. Make sure to check the specific requirements or guidelines provided by the university for each course before registering. If you encounter any issues or need further assistance, feel free to ask! I don't have that information in my knowledge base.

── Retrieved sources ──
  [1] course_registration.pdf: UNIVERSITY OF PERADENIYA — COURSE REGISTRATION GUIDE    Semester system:  The university operates on a two-semester acad…
  [2] course_registration.pdf: UNIVERSITY OF PERADENIYA — COURSE REGISTRATION GUIDE    Semester system:  The university operates on a two-semester acad…
  [3] admission_requirments.txt: UNIVERSITY OF P

...

A: To access IEEE Xplore from off campus, you need to download the GlobalProtect VPN client from itc.pdn.ac.lk/vpn. You will then connect to the central IT lab's GPU workstation via the internet. Once connected, you should be able to search for articles related to IEEE Xplore on the central server. If you encounter any issues or problems accessing the service, please contact IT Services for assistance. 

I don't have that information in my knowledge base.

── Retrieved sources ──
  [1] ITServices.pdf: Computer labs:  - Each faculty has at least one computer lab (open during working hours).  - The central IT lab (Buildin…
  [2] ITServices.pdf: Computer labs:  - Each faculty has at least one computer lab (open during working hours).  - The central IT lab (Buildin…
  [3] ITServices.pdf: UNIVERSITY OF PERADENIYA — IT SERVICES & INFRASTRUCTURE    Student email:  - Every enrolled student receives a universit…

Q: What financial aid options are available for students?
------------------------

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: The university offers two types of financial aid: bursaries and financial aid. Bursaries are government-funded scholarships provided based on income and academic merit, while financial aid includes Mahapola Scholarships which are government-funded and awarded based on income and academic merit. Additionally, there are postgraduate students who can apply for 8 books, 4-week loan, and academic staff members who can borrow 20 books and one semester loan. Lastly, remote access to digital resources is also available through the university network or VPN. There's no mention of turnitin as an option for plagiarism checks.

── Retrieved sources ──
  [1] student_welfare.pdf: UNIVERSITY OF PERADENIYA — STUDENT WELFARE & SUPPORT    Accommodation:  The university provides 17 residential halls (se…
  [2] student_welfare.pdf: UNIVERSITY OF PERADENIYA — STUDENT WELFARE & SUPPORT    Accommodation:  The university provides 17 residential halls (se…
  [3] library_services.pdf: Postgraduates   : 8 boo

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: Software licenses available to students (free) include:

1. MATLAB
2. AutoCAD
3. SPSS
4. Microsoft 365
5. GitHub Student Developer Pack

The other options listed do not fall under the category of free software licenses specifically designed for students. They are either related to specific services or resources provided by the university but are not considered free software licenses for students. 

I don't have that information in my knowledge base. Please provide more details if you need further assistance.

── Retrieved sources ──
  [1] ITServices.pdf: Computer labs:  - Each faculty has at least one computer lab (open during working hours).  - The central IT lab (Buildin…
  [2] ITServices.pdf: Computer labs:  - Each faculty has at least one computer lab (open during working hours).  - The central IT lab (Buildin…
  [3] ITServices.pdf: UNIVERSITY OF PERADENIYA — IT SERVICES & INFRASTRUCTURE    Student email:  - Every enrolled student receives a universit…

Q: What is the process fo

Reload vector store

In [ ]:
#Use the cell in future colab sessions so that we do not need to rebuild the vector store from scratch every time.
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name="university_docs",
)

print(f"Loaded {vectorstore._collection.count()} vectors from disk.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded 66 vectors from disk.


/tmp/ipykernel_1214/383547437.py:11: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


Interactive Chat Loop

In [ ]:
# If need to exit type "quit"
print("University RAG Assistant — type your question (or 'quit' to exit)\n")
while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    if not user_input:
        continue
    ask(user_input)

University RAG Assistant — type your question (or 'quit' to exit)

You: What is the format of student email?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the format of student email?
-----------------------------------------------------------------
A: The format of student email is studentID@stu.pdn.ac.lk. 

I don't have that information in my knowledge base.

── Retrieved sources ──
  [1] ITServices.pdf: UNIVERSITY OF PERADENIYA — IT SERVICES & INFRASTRUCTURE    Student email:  - Every enrolled student receives a universit…
  [2] ITServices.pdf: UNIVERSITY OF PERADENIYA — IT SERVICES & INFRASTRUCTURE    Student email:  - Every enrolled student receives a universit…
  [3] course_registration.pdf: UNIVERSITY OF PERADENIYA — COURSE REGISTRATION GUIDE    Semester system:  The university operates on a two-semester acad…

You: How to register for courses?


[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How to register for courses?
-----------------------------------------------------------------
A: To register for courses, follow these steps:

1. Log in to the Student Information System (SIS) at sis.pdn.ac.lk with your university email and password.
2. Navigate to "Course Registration" under the Academic tab. 

This guide provides clear instructions on how to complete the course registration process using the SIS platform. If you need further assistance or specific details about the registration process, please refer to the official University of Peradeniya website for more detailed information. I don't have that information in my knowledge base.

── Retrieved sources ──
  [1] course_registration.pdf: 3. Select courses for the upcoming semester within the registration window     (usually 2 weeks before semester start). …
  [2] course_registration.pdf: 3. Select courses for the upcoming semester within the registration window     (usually 2 weeks before semester start). …
  [3] cou

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What is the credit limit of postgraduate courswork
-----------------------------------------------------------------
A: The credit limit of postgraduate coursework is 12-15 credits per semester. This can be determined by referring to the context provided, which states that postgraduate coursework students may register for 12-15 credits per semester. However, it does not specify any specific credit limit for postgraduate coursework. Therefore, I do not have that information in my knowledge base. To provide an accurate response, additional context would be needed. 

I don't have that information in my knowledge base.

── Retrieved sources ──
  [1] course_registration.pdf: - Postgraduate coursework students may register for 12–15 credits per semester.    Adding / dropping courses:  - Add per…
  [2] course_registration.pdf: - Postgraduate coursework students may register for 12–15 credits per semester.    Adding / dropping courses:  - Add per…
  [3] course_registration.pdf: 3. Select co